# Python code for isochromes of the bicycle network in the municipality of Groningen

First, we install all the packages. We use matplotlib.pyplot to plot the figure on screen. We use osmnx to get the data from Open Street Map. We use networkx to calculate the fastest route and we use geopandas to form polygons for the isochromes. 

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
import geopandas as gpd

Next, we import the bicycle data from OSM to our pythoncode. We will only use the data of 'Groningen'.

In [2]:
ox.settings.use_cache = True
Groningen = ox.graph_from_place("Groningen, Netherlands", network_type="bike")

nodes, edges = ox.graph_to_gdfs(Groningen, nodes=True, edges=True)
nodes.head()
edges.head()

osmid  \
u        v           key                                                      
26983966 7798106394  0                                            835449636   
         11459644067 0    [75319593, 75319595, 1414845771, 726842958, 33...   
         5184074334  0                               [534589497, 835449637]   
26983968 891920815   0                                              6394958   
         5184791692  0                                            181926420   

                              highway                         name  oneway  \
u        v           key                                                     
26983966 7798106394  0       cycleway                   Heerdenpad   False   
         11459644067 0       cycleway  [Korreweg, Gerrit Krolbrug]   False   
         5184074334  0           path                  Ulgersmaweg   False   
26983968 891920815   0    residential                   Heerdenpad   False   
         5184791692  0    residential                   Heerdenpad   False   

                         width       reversed      length lanes bridge  \
u        v           key                                                 
26983966 7798106394  0       4          False    8.728186   NaN    NaN   
         11459644067 0     NaN  [False, True]  368.349111     2    yes   
         5184074334  0     NaN  [False, True]   69.123213   NaN    NaN   
26983968 891920815   0     NaN           True   58.437638   NaN    NaN   
         5184791692  0     NaN          False   32.952527   NaN    NaN   

                                                                   geometry  \
u        v           key                                                      
26983966 7798106394  0      LINESTRING (6.57866 53.23651, 6.57872 53.23658)   
         11459644067 0    LINESTRING (6.57866 53.23651, 6.57859 53.23643...   
         5184074334  0    LINESTRING (6.57866 53.23651, 6.57862 53.23653...   
26983968 891920815   0    LINESTRING (6.57801 53.23735, 6.57802 53.23726...   
         5184791692  0    LINESTRING (6.57801 53.23735, 6.57796 53.23758...   

                          maxspeed tunnel access service est_width  ref  \
u        v           key                                                  
26983966 7798106394  0         NaN    NaN    NaN     NaN       NaN  NaN   
         11459644067 0         NaN    NaN    NaN     NaN       NaN  NaN   
         5184074334  0    [30, 50]    NaN    NaN     NaN       NaN  NaN   
26983968 891920815   0          30    NaN    NaN     NaN       NaN  NaN   
         5184791692  0          30    NaN    NaN     NaN       NaN  NaN   

                         junction  
u        v           key           
26983966 7798106394  0        NaN  
         11459644067 0        NaN  
         5184074334  0        NaN  
26983968 891920815   0        NaN  
         5184791692  0        NaN

Next, we will use The main station of Groningen as the centre of our map. this is the point we will calculate the distamnces and thus time to cycle from. We determine the coordinates of this point. After that we look for the point that is closest to this coordinate, which is on the network. 

In [3]:
lat, lon = ox.geocode("Hoofdstation, Groningen")
main_station = ox.distance.nearest_nodes(Groningen, X=lon, Y=lat)

print(f'latitude(x) :{lat} and longitude(y): {lon}')
print(main_station)

latitude(x) :53.210941 and longitude(y): 6.564096
893144394


Step 3: determine the distances and travel time. First, we will change the coordinate system to meters using project_graph. Then we will use an average speed of 15 km/h ( = 15*1000/60 meter/min) and add the time to the dataframe.  

In [4]:
Groningen = ox.project_graph(Groningen)
meters_per_min = 15 * 1000 / 60
for start_node, end_node, key, data in Groningen.edges(keys=True, data=True):
    data["time"] = data["length"] / meters_per_min


Step 4: We make a GeoDataFrame of the isochromes. First we determine our desired distances. (trip_times). next we make an empty list to store the polygons. we loop over the trip_times and for each trip_time we first detemine if each node is in or outside of te traveltime. if its inside, we add it to the polygon. At the end we have multiple polygons and store this in a geodataframe.

In [5]:
trip_times = [5, 10, 15, 20, 25]
polygons = []
for t in sorted(trip_times, reverse=True):
    sub = nx.ego_graph(Groningen, main_station, radius=t, distance="time")
    node = ox.graph_to_gdfs(sub, edges=False)
    poly = node.geometry.buffer(100).union_all()
    polygons.append({"minuten": t, "geometry": poly})

iso = gpd.GeoDataFrame(polygons, crs=Groningen.graph["crs"])

print(iso)

   minuten                                           geometry
0       25  MULTIPOLYGON (((333390.795 5897575.197, 333381...
1       20  MULTIPOLYGON (((333712.712 5897124.113, 333703...
2       15  MULTIPOLYGON (((334866.748 5896895.254, 334857...
3       10  POLYGON ((335334.228 5897846.918, 335325.357 5...
4        5  POLYGON ((336389.47 5898403.548, 336379.951 58...


Step 5: visualising.

In [6]:
maps = iso.explore(column="minuten", cmap="plasma", tiles="OpenStreetMap", style_kwds={"fillOpacity": 0.3})

#edges.explore(m=maps, color="red", style_kwds={"weight": 1})
#nodes.explore(m=maps, color="black", style_kwds={"weight": 1})

municipality = ox.geocode_to_gdf("Groningen, Netherlands")
municipality.boundary.explore(m=maps, color="black")

maps

In [7]:
maps.save("Isochrone bycicle network.html")